# Successful vs Unsuccessful Grant Comparison

Run the scoring pipeline on all PDFs in `data/successful/` and `data/unsuccessful/`, print results, and export to Excel.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('OLLAMA_HOST', 'http://localhost:11435')
os.environ.setdefault('OLLAMA_NUM_CTX', '80000')   # 大 context window 给 baseline 用

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save
from src.pool.build_pool import build_chunk_pool
from src.scoring.pipeline import (
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
    _aggregate_overall,
    _aggregate_section,
    _build_scored_section,
    _generate_json_with_parse_retry,
    _normalize_model_section_output,
    load_rubric,
)

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
SUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'successful'
UNSUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'unsuccessful'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results' / 'compare'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'
BASELINE_MAX_TOKENS = int(os.environ.get('WHOLE_JSON_BASELINE_MAX_TOKENS', '65536'))

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

for path in [RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root      :', PROJECT_ROOT)
print('Results dir       :', RESULTS_DIR)
print('Ollama host       :', os.environ.get('OLLAMA_HOST'))
print('Ollama num_ctx    :', os.environ.get('OLLAMA_NUM_CTX'))
print('Model             :', EXPERIMENT_OLLAMA_MODEL)
print('Baseline max tok  :', BASELINE_MAX_TOKENS)

In [ ]:
def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    raw = json_path.read_text(encoding='utf-8').strip()
    if not raw:
        raise ValueError(f'Parsed JSON is empty for {pdf_path.name} — parser may have failed silently.')
    parsed = json.loads(raw)
    return parsed, json_path


def score_pdf(pdf_path: Path, *, scorer: _Scorer, reparse: bool = False) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / 'artifacts' / pdf_path.stem
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed, CRITERIA_PATH,
        doc_id=pdf_path.stem, scorer=scorer, artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    return result


def _compact_text(text: str, limit: int = 900) -> str:
    compact = ' '.join((text or '').split())
    return compact[:limit] + '...' if len(compact) > limit else compact


def score_pdf_baseline(pdf_path: Path, *, scorer: _Scorer) -> dict:
    """One-shot baseline: full parsed JSON input, per-sub-criterion 0-10 scoring."""
    parsed, _ = parse_pdf_cached(pdf_path)
    rubric_sections = load_rubric(CRITERIA_PATH)
    pool_data = build_chunk_pool(parsed)
    pool_lookup = pool_data['pool_lookup']
    all_chunk_ids = list(pool_lookup)

    chunk_index = [
        {
            'chunk_id': cid,
            'parser_section': m.get('parser_section', ''),
            'preview': _compact_text(m.get('text', '')),
        }
        for cid, m in pool_lookup.items()
    ]
    criteria_simplified = [
        {
            'section_key': sec['section_key'],
            'section_name': sec['human_name'],
            'sub_criteria': [
                {'sub_id': sub['sub_id'], 'name': sub['name'],
                 'signals': [s['text'] for s in sub['signals']]}
                for sub in sec['sub_criteria']
            ],
        }
        for sec in rubric_sections
    ]

    chunk_id_item = {'type': 'string', 'enum': all_chunk_ids} if all_chunk_ids else {'type': 'string'}
    sec_props = {
        sec['section_key']: {
            'type': 'object',
            'properties': {
                sub['sub_id']: {
                    'type': 'object',
                    'properties': {
                        'score': {'type': 'integer', 'enum': list(range(11))},
                        'not_applicable': {'type': 'boolean'},
                        'used_chunk_ids': {'type': 'array', 'items': chunk_id_item, 'maxItems': 5},
                        'rationale': {'type': 'string'},
                    },
                    'required': ['score', 'not_applicable', 'used_chunk_ids', 'rationale'],
                    'additionalProperties': False,
                }
                for sub in sec['sub_criteria']
            },
            'required': [sub['sub_id'] for sub in sec['sub_criteria']],
            'additionalProperties': False,
        }
        for sec in rubric_sections
    }
    schema = {'type': 'object', 'properties': sec_props,
              'required': list(sec_props), 'additionalProperties': False}

    example_cid = all_chunk_ids[0] if all_chunk_ids else 'chunk_id'
    example = {
        'general': {
            'g.1': {'score': 8, 'not_applicable': False, 'used_chunk_ids': [example_cid], 'rationale': 'Strong vision...'},
            'g.2': {'score': 0, 'not_applicable': True, 'used_chunk_ids': [], 'rationale': 'Not applicable for this grant type.'},
        }
    }

    messages = [
        {
            'role': 'system',
            'content': (
                'You are a strict NIHR grant scoring baseline. You do not use staged retrieval, '
                'belief state, or multi-pass reasoning. Return JSON only and follow the schema exactly.\n'
                'For each sub-criterion, score from 0 to 10 strictly based on the signals listed — '
                'each signal represents a specific scoring dimension. '
                'Assess the evidence for every signal individually, then derive your 0-10 score from their combined assessment. '
                'A score of 10 requires strong evidence for all signals; partial evidence yields a proportionally lower score. '
                'If a sub-criterion is not applicable to this grant type, set not_applicable to true and score to 0. '
                'List up to 5 chunk_ids that directly support your score.\n'
                'Return JSON with this exact structure:\n'
                + json.dumps(example, indent=2)
            ),
        },
        {
            'role': 'user',
            'content': json.dumps({
                'criteria': criteria_simplified,
                'evidence_chunk_index': chunk_index,
                'parsed_application_json': parsed,
            }, ensure_ascii=False),
        },
    ]

    _, parsed_response, _ = _generate_json_with_parse_retry(
        scorer, messages, schema=schema, max_tokens=BASELINE_MAX_TOKENS, max_retries=0,
    )

    # Simple average per section, excluding not_applicable
    features: dict = {}
    all_scores: list[float] = []
    for sec in rubric_sections:
        sk = sec['section_key']
        sec_data = (parsed_response or {}).get(sk, {})
        if not isinstance(sec_data, dict):
            sec_data = {}
        sec_scores, sec_evidence = [], 0
        for sub in sec['sub_criteria']:
            sub_data = sec_data.get(sub['sub_id'], {})
            if isinstance(sub_data, dict) and not sub_data.get('not_applicable'):
                sec_scores.append(float(sub_data.get('score', 0)))
                sec_evidence += len(sub_data.get('used_chunk_ids', []))
        sec_avg = sum(sec_scores) / len(sec_scores) if sec_scores else 0.0
        features[sk] = {
            'overall': {
                'score_10': round(sec_avg, 2),
                'final_score_0to100': round(sec_avg * 10, 2),
                'evidence_count': sec_evidence,
            }
        }
        all_scores.extend(sec_scores)

    overall_avg = sum(all_scores) / len(all_scores) if all_scores else 0.0
    return {
        'overall': {
            'score_10': round(overall_avg, 2),
            'final_score_0to100': round(overall_avg * 10, 2),
        },
        'features': features,
    }


def flatten_scores(result: dict, *, prefix: str) -> dict:
    overall = result.get('overall', {})
    row: dict = {f'{prefix}overall_score_100': round(float(overall.get('final_score_0to100', 0)), 2)}
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        sec = features.get(section_key, {}).get('overall', {})
        row[f'{prefix}{section_key}_score_100'] = round(float(sec.get('final_score_0to100', 0)), 2)
        row[f'{prefix}{section_key}_evidence_count'] = int(sec.get('evidence_count', 0))
    return row


print('Helper functions defined.')

In [ ]:
# ── Run PIPELINE: SUCCESSFUL ──────────────────────────────────────────────────
PIPELINE_CSV_S = RESULTS_DIR / 'progress_pipeline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))

done_ps = set(pd.read_csv(PIPELINE_CSV_S)['pdf_name'].tolist()) if PIPELINE_CSV_S.exists() else set()
print(f'Pipeline successful: {len(done_ps)}/{len(successful_pdfs)} already done')

failed_ps: list[tuple[str, str]] = []

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    if pdf_path.name in done_ps:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}')
        continue

    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        row = {'pdf_name': pdf_path.name, 'category': 'successful'}
        row.update(flatten_scores(score_pdf(pdf_path, scorer=scorer), prefix='pipeline_'))
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_S, mode='a', header=not PIPELINE_CSV_S.exists(), index=False)
        done_ps.add(pdf_path.name)
        print(f'  pipeline overall={row["pipeline_overall_score_100"]:.1f}')
        for k in SECTION_KEYS:
            print(f'    {k:<25} {row[f"pipeline_{k}_score_100"]:5.1f} (ev={row[f"pipeline_{k}_evidence_count"]})')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed_ps.append((pdf_path.name, str(exc)))

print(f'\nDone. {len(done_ps)}/{len(successful_pdfs)} successful PDFs (pipeline)')
if failed_ps:
    for name, err in failed_ps:
        print(f'  FAILED - {name}: {err}')

In [ ]:
# ── Run BASELINE: SUCCESSFUL ──────────────────────────────────────────────────
BASELINE_CSV_S = RESULTS_DIR / 'progress_baseline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))

done_bs = set(pd.read_csv(BASELINE_CSV_S)['pdf_name'].tolist()) if BASELINE_CSV_S.exists() else set()
print(f'Baseline successful: {len(done_bs)}/{len(successful_pdfs)} already done')

failed_bs: list[tuple[str, str]] = []

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    if pdf_path.name in done_bs:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}')
        continue

    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        row = {'pdf_name': pdf_path.name, 'category': 'successful'}
        row.update(flatten_scores(score_pdf_baseline(pdf_path, scorer=scorer), prefix='baseline_'))
        pd.DataFrame([row]).to_csv(BASELINE_CSV_S, mode='a', header=not BASELINE_CSV_S.exists(), index=False)
        done_bs.add(pdf_path.name)
        print(f'  baseline overall={row["baseline_overall_score_100"]:.1f}')
        for k in SECTION_KEYS:
            print(f'    {k:<25} {row[f"baseline_{k}_score_100"]:5.1f} (ev={row[f"baseline_{k}_evidence_count"]})')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed_bs.append((pdf_path.name, str(exc)))

print(f'\nDone. {len(done_bs)}/{len(successful_pdfs)} successful PDFs (baseline)')
if failed_bs:
    for name, err in failed_bs:
        print(f'  FAILED - {name}: {err}')

In [ ]:
# ── Export SUCCESSFUL Excel (merge pipeline + baseline) ───────────────────────
section_cols = []
for k in SECTION_KEYS:
    section_cols += [
        f'pipeline_{k}_score_100', f'baseline_{k}_score_100',
        f'pipeline_{k}_evidence_count', f'baseline_{k}_evidence_count',
    ]
display_cols = ['pdf_name', 'pipeline_overall_score_100', 'baseline_overall_score_100', *section_cols]

df_p = pd.read_csv(PIPELINE_CSV_S)
df_b = pd.read_csv(BASELINE_CSV_S).drop(columns=['category'], errors='ignore')
df_s = df_p.merge(df_b, on='pdf_name', how='outer').sort_values('pdf_name').reset_index(drop=True)

available_cols = [c for c in display_cols if c in df_s.columns]
display(df_s[available_cols])

ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
excel_s = RESULTS_DIR / f'successful_{ts}.xlsx'
df_s[available_cols].to_excel(excel_s, index=False, sheet_name='Successful')
print(f'Exported: {excel_s}  ({len(df_s)} rows)')

In [ ]:
# ── Run PIPELINE: UNSUCCESSFUL ────────────────────────────────────────────────
PIPELINE_CSV_U = RESULTS_DIR / 'progress_pipeline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))

done_pu = set(pd.read_csv(PIPELINE_CSV_U)['pdf_name'].tolist()) if PIPELINE_CSV_U.exists() else set()
print(f'Pipeline unsuccessful: {len(done_pu)}/{len(unsuccessful_pdfs)} already done')

failed_pu: list[tuple[str, str]] = []

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    if pdf_path.name in done_pu:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}')
        continue

    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        row = {'pdf_name': pdf_path.name, 'category': 'unsuccessful'}
        row.update(flatten_scores(score_pdf(pdf_path, scorer=scorer), prefix='pipeline_'))
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_U, mode='a', header=not PIPELINE_CSV_U.exists(), index=False)
        done_pu.add(pdf_path.name)
        print(f'  pipeline overall={row["pipeline_overall_score_100"]:.1f}')
        for k in SECTION_KEYS:
            print(f'    {k:<25} {row[f"pipeline_{k}_score_100"]:5.1f} (ev={row[f"pipeline_{k}_evidence_count"]})')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed_pu.append((pdf_path.name, str(exc)))

print(f'\nDone. {len(done_pu)}/{len(unsuccessful_pdfs)} unsuccessful PDFs (pipeline)')
if failed_pu:
    for name, err in failed_pu:
        print(f'  FAILED - {name}: {err}')

In [ ]:
# ── Run BASELINE: UNSUCCESSFUL ────────────────────────────────────────────────
BASELINE_CSV_U = RESULTS_DIR / 'progress_baseline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))

done_bu = set(pd.read_csv(BASELINE_CSV_U)['pdf_name'].tolist()) if BASELINE_CSV_U.exists() else set()
print(f'Baseline unsuccessful: {len(done_bu)}/{len(unsuccessful_pdfs)} already done')

failed_bu: list[tuple[str, str]] = []

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    if pdf_path.name in done_bu:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}')
        continue

    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        row = {'pdf_name': pdf_path.name, 'category': 'unsuccessful'}
        row.update(flatten_scores(score_pdf_baseline(pdf_path, scorer=scorer), prefix='baseline_'))
        pd.DataFrame([row]).to_csv(BASELINE_CSV_U, mode='a', header=not BASELINE_CSV_U.exists(), index=False)
        done_bu.add(pdf_path.name)
        print(f'  baseline overall={row["baseline_overall_score_100"]:.1f}')
        for k in SECTION_KEYS:
            print(f'    {k:<25} {row[f"baseline_{k}_score_100"]:5.1f} (ev={row[f"baseline_{k}_evidence_count"]})')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed_bu.append((pdf_path.name, str(exc)))

print(f'\nDone. {len(done_bu)}/{len(unsuccessful_pdfs)} unsuccessful PDFs (baseline)')
if failed_bu:
    for name, err in failed_bu:
        print(f'  FAILED - {name}: {err}')

In [ ]:
# ── Export UNSUCCESSFUL Excel (merge pipeline + baseline) ─────────────────────
section_cols = []
for k in SECTION_KEYS:
    section_cols += [
        f'pipeline_{k}_score_100', f'baseline_{k}_score_100',
        f'pipeline_{k}_evidence_count', f'baseline_{k}_evidence_count',
    ]
display_cols = ['pdf_name', 'pipeline_overall_score_100', 'baseline_overall_score_100', *section_cols]

df_p = pd.read_csv(PIPELINE_CSV_U)
df_b = pd.read_csv(BASELINE_CSV_U).drop(columns=['category'], errors='ignore')
df_u = df_p.merge(df_b, on='pdf_name', how='outer').sort_values('pdf_name').reset_index(drop=True)

available_cols = [c for c in display_cols if c in df_u.columns]
display(df_u[available_cols])

ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
excel_u = RESULTS_DIR / f'unsuccessful_{ts}.xlsx'
df_u[available_cols].to_excel(excel_u, index=False, sheet_name='Unsuccessful')
print(f'Exported: {excel_u}  ({len(df_u)} rows)')